# Training/serving skew and leakage, demonstrated

This notebook runs two leakage experiments on simulated fraud data that has a time
column. Each experiment trains the same logistic regression twice, once with an
honest pipeline and once with a leaky one, and prints the test AUC of both.

- **Demo 1** matches `notes.md` Part C1, cause 3 (different preprocessing
  statistics): a scaler fit on all data, including the future test period,
  before a temporal split.
- **Demo 2** matches `notes.md` Part C2 (point-in-time correctness): a "card's
  transaction count today" feature computed over the whole day, which includes
  transactions that happen after the one being scored.

Only numpy and scikit-learn are used; the seed is fixed and the whole notebook
runs in a few seconds on a CPU. Run it from the repo root with
`uv run jupyter execute ml/06_ml_systems/feature_pipeline_demo.ipynb`.

## Setup

The next cell imports the libraries and creates one random generator with a fixed
seed. Both demos draw from this generator in order, so the printed numbers are the
same on every run. It also defines two helpers: `fit_lr` fits a logistic
regression, and `auc` returns the ROC AUC of a fitted model on a labeled dataset.

In [1]:
from collections import defaultdict

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

RNG = np.random.default_rng(seed=7)


def fit_lr(X, y):
    model = LogisticRegression(max_iter=2000)
    model.fit(X, y)
    return model


def auc(model, X, y):
    return roc_auc_score(y, model.predict_proba(X)[:, 1])

## Demo 1: scaler fit on all data vs fit on the training period only

The next cell simulates 40,000 transactions at times $t \in [0, 1]$ with a 5% fraud
rate. The informative feature, an "amount anomaly" score, has a baseline that rises
as $10t^3$, and fraud rows sit 1.5 units above legitimate rows at every moment; two
pure-noise features are added. The data is split by time (the first 70% is
training). A min-max scaler with clipping is then fit two ways: on all rows
(leaky, because it has seen the test period's range) and on training rows only
(honest, because that is the only scaler production can have).

Expect the leaky scaler to give the higher test AUC. Under the honest scaler,
about half of the test rows exceed the training maximum and are clipped to the
same value, which removes the fraud offset for those rows.

In [2]:
def demo_scaler_leak():
    n = 40_000
    t = np.sort(RNG.uniform(0.0, 1.0, size=n))          # time in [0, 1]
    y = (RNG.uniform(size=n) < 0.05).astype(int)         # 5% fraud

    # Informative feature: an "amount anomaly" score whose baseline rises as
    # 10 * t^3, so the test period reaches values the training period never
    # saw. Fraud sits a fixed offset above legit at any moment in time.
    drift = 10.0 * t**3
    noise = RNG.uniform(-2.0, 2.0, size=n)               # bounded noise
    x_amt = drift + 1.5 * y + noise                      # fraud offset: +1.5
    x_noise1 = RNG.normal(0.0, 1.0, size=n)
    x_noise2 = RNG.normal(0.0, 1.0, size=n)
    X = np.column_stack([x_amt, x_noise1, x_noise2])

    train = t < 0.70                                     # temporal split
    test = ~train
    X_tr, y_tr = X[train], y[train]
    X_te, y_te = X[test], y[test]

    # LEAKY: scaler fit on all rows, past and future, THEN split.
    leaky_scaler = MinMaxScaler(clip=True).fit(X)
    m_leaky = fit_lr(leaky_scaler.transform(X_tr), y_tr)
    auc_leaky = auc(m_leaky, leaky_scaler.transform(X_te), y_te)

    # HONEST: scaler fit on the training period only. Test values above the
    # training range are clipped to 1.0, so fraud and legit rows in that range
    # get the same scaled value.
    honest_scaler = MinMaxScaler(clip=True).fit(X_tr)
    m_honest = fit_lr(honest_scaler.transform(X_tr), y_tr)
    auc_honest = auc(m_honest, honest_scaler.transform(X_te), y_te)

    frac_clip = (X_te[:, 0] > X_tr[:, 0].max()).mean()
    print("Demo 1: scaler fit on ALL data vs fit on TRAIN only (temporal split)")
    print(f"  train rows: {train.sum():>6}   test rows: {test.sum():>6}   "
          f"fraud rate: {y.mean():.3f}")
    print(f"  test rows whose amount exceeds the training-period max: "
          f"{frac_clip:.1%}")
    print(f"  offline test AUC, LEAKY  scaler (fit on all data) : {auc_leaky:.4f}")
    print(f"  offline test AUC, HONEST scaler (fit on train)    : {auc_honest:.4f}")
    print(f"  inflation from the leak                            : "
          f"+{auc_leaky - auc_honest:.4f}")
    print("  The leaky number is unattainable in production: the serving-time")
    print("  scaler cannot have been fit on data from the future.")
    return auc_leaky, auc_honest, frac_clip


d1_leaky, d1_honest, d1_frac_clip = demo_scaler_leak()

# Checks on the claims in the markdown above.
assert d1_leaky > d1_honest, "the leaky scaler should inflate offline AUC"
assert d1_frac_clip > 0.3, "a large share of test rows should fall outside the training range"

Demo 1: scaler fit on ALL data vs fit on TRAIN only (temporal split)
  train rows:  27831   test rows:  12169   fraud rate: 0.048
  test rows whose amount exceeds the training-period max: 45.9%
  offline test AUC, LEAKY  scaler (fit on all data) : 0.6900
  offline test AUC, HONEST scaler (fit on train)    : 0.6583
  inflation from the leak                            : +0.0316
  The leaky number is unattainable in production: the serving-time
  scaler cannot have been fit on data from the future.


## Demo 2, step 1: simulate card transactions and build the two count features

The next cell simulates 4,000 cards over 30 days. Each card makes legitimate
transactions at about 0.4 per day. 6% of cards are compromised at a random time,
after which 4 to 12 fraudulent transactions occur within half a day.

For every transaction it then computes two versions of "transactions by this card
today". The **leaky** count covers the whole calendar day, including transactions
after the one being scored. The **point-in-time** count covers only earlier
transactions that day, which is all that exists at decision time. A brute-force
check on a sample of rows confirms that the point-in-time count uses only earlier
transactions. Expect the leaky count to be much larger than the point-in-time
count on fraud rows, because it includes the rest of the burst.

In [3]:
def make_transactions():
    # A card's full-day count on a fraud day includes the burst transactions
    # that happen AFTER the one being scored.
    n_cards, n_days = 4000, 30
    rows = []  # (card, time, y, amount, mismatch)
    for card in range(n_cards):
        compromised = RNG.uniform() < 0.06
        t_comp = RNG.uniform(0, n_days) if compromised else np.inf
        # normal activity: ~0.4 transactions/day
        n_legit = RNG.poisson(0.4 * n_days)
        for time in RNG.uniform(0, n_days, size=n_legit):
            amount = RNG.lognormal(3.0, 0.7)
            mismatch = RNG.uniform() < 0.05
            rows.append((card, time, 0, amount, mismatch))
        # fraud burst: 4 to 12 transactions within half a day of compromise
        if compromised:
            n_burst = RNG.integers(4, 13)
            for time in t_comp + RNG.uniform(0, 0.5, size=n_burst):
                if time < n_days:
                    amount = RNG.lognormal(3.4, 0.7)   # only slightly larger
                    mismatch = RNG.uniform() < 0.25    # only mildly telling
                    rows.append((card, time, 1, amount, mismatch))
    rows.sort(key=lambda r: r[1])
    card = np.array([r[0] for r in rows])
    time = np.array([r[1] for r in rows])
    y = np.array([r[2] for r in rows])
    amount = np.array([r[3] for r in rows])
    mismatch = np.array([r[4] for r in rows], dtype=float)
    return card, time, y, amount, mismatch


def day_count_features(card, time):
    # leaky   = other transactions by this card on the same calendar day
    #           (past AND future)
    # correct = transactions by this card earlier on the same day (past only)
    day = np.floor(time).astype(int)
    n = len(card)
    leaky = np.zeros(n)
    correct = np.zeros(n)
    per_key = defaultdict(list)          # (card, day) -> row indices
    for i in range(n):
        per_key[(card[i], day[i])].append(i)
    for idx in per_key.values():
        total = len(idx)
        # rows are globally time-sorted, so idx is time-ordered within the key
        for rank, i in enumerate(idx):
            leaky[i] = total - 1
            correct[i] = rank
    return leaky, correct


card, time, y, amount, mismatch = make_transactions()
leaky_cnt, correct_cnt = day_count_features(card, time)

# Brute-force check on every 250th row (deterministic, uses no random draws):
# the point-in-time count equals the number of same-card, same-day
# transactions strictly earlier in time.
day = np.floor(time).astype(int)
for i in range(0, len(card), 250):
    same = (card == card[i]) & (day == day[i])
    assert correct_cnt[i] == np.sum(same & (time < time[i]))
    assert leaky_cnt[i] == np.sum(same) - 1
assert np.all(correct_cnt <= leaky_cnt)

print(f"transactions: {len(y)}   fraud rate: {y.mean():.3f}")
print(f"{'mean count today':<28}{'legit rows':>12}{'fraud rows':>12}")
print(f"{'leaky (whole day)':<28}{leaky_cnt[y == 0].mean():>12.2f}"
      f"{leaky_cnt[y == 1].mean():>12.2f}")
print(f"{'point-in-time (earlier)':<28}{correct_cnt[y == 0].mean():>12.2f}"
      f"{correct_cnt[y == 1].mean():>12.2f}")
assert leaky_cnt[y == 1].mean() > correct_cnt[y == 1].mean() + 1.0

transactions: 49992   fraud rate: 0.040
mean count today              legit rows  fraud rows
leaky (whole day)                   0.41        6.70
point-in-time (earlier)             0.21        3.34


## Demo 2, step 2: train and evaluate with each version of the feature

The next cell splits by time (days 0 to 24 are training, days 24 to 30 are test)
and fits logistic regressions on log(amount), an address-mismatch flag, and
optionally one of the two counts. It reports four test AUCs:

1. base features only, no count;
2. honest: point-in-time count at training and at test;
3. leaky offline: full-day count at training and at test, which is what an
   offline evaluation on the complete log would report;
4. deployed: trained on the full-day count but served the point-in-time count,
   because that is the only version production can compute at decision time.

Expect the leaky offline AUC to be the highest of the four, and the deployed AUC
to drop back to about the honest level. The gap between them (about 0.05 here) is
how much the leaky offline evaluation overstates production performance.

In [4]:
def demo_point_in_time(time, y, amount, mismatch, leaky_cnt, correct_cnt):
    base = np.column_stack([np.log(amount), mismatch])
    X_leaky = np.column_stack([base, leaky_cnt])
    X_correct = np.column_stack([base, correct_cnt])

    train = time < 24.0                                  # temporal split
    test = ~train
    y_tr, y_te = y[train], y[test]

    def run(X_train_feats, X_test_feats):
        scaler = StandardScaler().fit(X_train_feats[train])  # honest scaler
        m = fit_lr(scaler.transform(X_train_feats[train]), y_tr)
        return auc(m, scaler.transform(X_test_feats[test]), y_te)

    auc_base = run(base, base)
    auc_honest = run(X_correct, X_correct)
    # Offline, the full-day count CAN be computed (the whole log exists).
    auc_leaky = run(X_leaky, X_leaky)
    # Production can only compute the past-only count at decision time.
    auc_deployed = run(X_leaky, X_correct)

    print("Demo 2: point-in-time-incorrect aggregate ('card's txn count today')")
    print(f"  train rows: {train.sum()}   test rows: {test.sum()}")
    print(f"  offline test AUC, base features only               : {auc_base:.4f}")
    print(f"  offline test AUC, HONEST past-only count           : {auc_honest:.4f}")
    print(f"  offline test AUC, LEAKY full-day count             : {auc_leaky:.4f}"
          "   <- highest, unattainable in production")
    print(f"  DEPLOYED: trained on leaky, served past-only count : {auc_deployed:.4f}"
          "   <- what production would see")
    print(f"  leaky offline AUC minus deployed AUC               : "
          f"{auc_leaky - auc_deployed:+.4f}")
    print(f"  deployed model minus honest model                  : "
          f"{auc_deployed - auc_honest:+.4f}")
    print("  The fraud burst continues AFTER the transaction being scored, so the")
    print("  full-day count partly encodes the label. At serving time only earlier")
    print("  transactions exist, so the deployed model performs at about the")
    print("  honest level, well below its leaky offline estimate.")
    return auc_base, auc_honest, auc_leaky, auc_deployed


d2_base, d2_honest, d2_leaky, d2_deployed = demo_point_in_time(
    time, y, amount, mismatch, leaky_cnt, correct_cnt)

# Checks on the claims in the markdown above.
assert d2_leaky > max(d2_base, d2_honest, d2_deployed), "leaky offline AUC should be highest"
assert d2_leaky - d2_deployed > 0.02, "offline estimate should overstate production"
assert d2_honest > d2_base, "the honest past-only count should still add signal"

Demo 2: point-in-time-incorrect aggregate ('card's txn count today')
  train rows: 40087   test rows: 9905
  offline test AUC, base features only               : 0.7246
  offline test AUC, HONEST past-only count           : 0.9372
  offline test AUC, LEAKY full-day count             : 0.9903   <- highest, unattainable in production
  DEPLOYED: trained on leaky, served past-only count : 0.9367   <- what production would see
  leaky offline AUC minus deployed AUC               : +0.0536
  deployed model minus honest model                  : -0.0005
  The fraud burst continues AFTER the transaction being scored, so the
  full-day count partly encodes the label. At serving time only earlier
  transactions exist, so the deployed model performs at about the
  honest level, well below its leaky offline estimate.


## Summary

The last cell collects the six AUCs into one table. In both demos the leaky
pipeline produces the higher offline number, while the number production would
actually get is the honest one (in Demo 2, the deployed model's AUC, which is
about equal to the honest model's).

In [5]:
print(f"  {'pipeline':<44}{'offline AUC':>12}")
print(f"  {'-' * 56}")
for name, value in [
    ("Demo 1  scaler fit on train only (honest)", d1_honest),
    ("Demo 1  scaler fit on all data (leaky)", d1_leaky),
    ("Demo 2  base features, no aggregate", d2_base),
    ("Demo 2  past-only aggregate (honest)", d2_honest),
    ("Demo 2  full-day aggregate (leaky)", d2_leaky),
    ("Demo 2  leaky-trained model in production", d2_deployed),
]:
    print(f"  {name:<44}{value:>12.4f}")
print()
print("  Both leaks raise the OFFLINE number without improving the PRODUCTION")
print("  model. If a new feature or pipeline change improves offline metrics by")
print("  a suspiciously large amount, first ask: could this computation have")
print("  been done at serving time, using only data from strictly before the")
print("  prediction moment?")

  pipeline                                     offline AUC
  --------------------------------------------------------
  Demo 1  scaler fit on train only (honest)         0.6583
  Demo 1  scaler fit on all data (leaky)            0.6900
  Demo 2  base features, no aggregate               0.7246
  Demo 2  past-only aggregate (honest)              0.9372
  Demo 2  full-day aggregate (leaky)                0.9903
  Demo 2  leaky-trained model in production         0.9367

  Both leaks raise the OFFLINE number without improving the PRODUCTION
  model. If a new feature or pipeline change improves offline metrics by
  a suspiciously large amount, first ask: could this computation have
  been done at serving time, using only data from strictly before the
  prediction moment?
